# 1. - Cкачиваем обученные модели и репозиторий

In [ ]:
%cd /content
!rm -rf HW-2_env
!git clone https://github.com/TebelevGt/HW-2_env.git
%cd HW-2_env/rl-shortest-path-agent

# 2. - Логика импортов

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
#%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from envs import (
    get_shortest_path_dataset, 
    correctness_reward_func, 
    reasoning_length_reward_func, 
    format_reward_func
)

# 3. - Обучение

In [ ]:
def train_grpo_model(
    dataset_path: str, 
    output_dir: str = "/kaggle/working/grpo_model_curiculum",
    max_steps: int = 250,
    num_generations: int = 8
):
    """
    Обучает модель Qwen2.5-1.5B методом GRPO на заданном датасете.
    """
    
    # --- 1. Конфигурация модели ---
    max_seq_length = 4096 * 2
    lora_rank = 64

    print(f"--- Загрузка модели и токенизатора ---")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen2.5-1.5B-Instruct",
        max_seq_length = max_seq_length,
        load_in_4bit = True,
        fast_inference = True,
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.8,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = lora_rank * 2,
        lora_dropout = 0.05,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = True,
    )

    tokenizer.padding_side = "left"

    # --- 2. Настройка GRPO ---
    print(f"--- Настройка GRPO Trainer (Dataset: {dataset_path}) ---")
    training_args = GRPOConfig(
        use_vllm = True, 
        learning_rate = 5e-6,
        adam_beta1 = 0.9,
        adam_beta2 = 0.99,
        weight_decay = 0.1,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        logging_steps = 1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        num_generations = num_generations, 
        max_prompt_length = 256,
        max_completion_length = 200,
        max_steps = max_steps,
        save_steps = max_steps,
        max_grad_norm = 0.1,
        report_to = "none",
        output_dir = "outputs",
        shuffle_dataset = False
    )

    trainer = GRPOTrainer(
        model = model,
        processing_class = tokenizer,
        reward_funcs = [
            correctness_reward_func,
            reasoning_length_reward_func,
            format_reward_func
        ],
        args = training_args,
        train_dataset = get_shortest_path_dataset(dataset_path)
    )

    # --- 3. Обучение и сохранение ---
    print(f"--- Начало обучения ---")
    trainer.train()
    
    print(f"--- Сохранение модели в {output_dir} ---")
    trainer.save_model(output_dir)
    
    return model, tokenizer

# Пример вызова:
model, tokenizer = train_grpo_model('data/train_curriculum.pkl')